# Spike Cross-Validation

80/20 train/test split, fit models across lasso grid, generate shrinkage trace plots.

In [ ]:
config_path = "config/config.yaml"
profile = None
run_name = None

In [ ]:
import warnings
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import multidms

from notebooks._common import load_config

In [ ]:
cfg = load_config(config_path, profile, run_name=run_name)
spike = cfg["spike"]
fit = spike["fitting"]

output_dir = spike["output_dir"]
reference = spike["reference"]
condition_colors = spike["condition_colors"]
chosen_lasso_strength = spike["chosen_lasso_strength"]
train_frac = cfg["train_frac"]
seed = spike.get("seed", cfg["seed"])
gpu_ids = cfg["gpu_ids"]
n_processes = cfg["n_processes"]

warnings.simplefilter("ignore")
%matplotlib inline
plt.rcParams.update({"legend.frameon": False, "font.size": 11})

In [ ]:
func_score_df = pd.read_csv(f"{output_dir}/training_functional_scores.csv").fillna({"aa_substitutions": ""})
models = pickle.load(open(f"{output_dir}/full_models.pkl", "rb"))

subsample_frac = spike.get("subsample_frac", None)
if subsample_frac is not None:
    func_score_df = (
        pd.concat([
        g.sample(frac=subsample_frac, random_state=0)
        for _, g in func_score_df.groupby(["condition", "replicate"])
    ]).reset_index(drop=True)
    )
print(f"Loaded {len(func_score_df)} variants, {len(models)} models")

## Train/test split

In [ ]:
cc = condition_colors
train, test = [], {}
for replicate, fs_df in func_score_df.groupby("replicate"):
    df_agg = (
        fs_df.groupby(["condition", "aa_substitutions"], dropna=False)
        .agg({"func_score": "mean"})
        .reset_index()
    )
    df_agg["aa_substitutions"] = df_agg["aa_substitutions"].fillna("")

    wt_mask = df_agg["aa_substitutions"].str.strip() == ""
    wt_rows = df_agg[wt_mask]
    non_wt = df_agg[~wt_mask].sample(frac=1, random_state=seed)
    n_train = int(len(non_wt) * train_frac)
    train_split = pd.concat([wt_rows, non_wt.iloc[:n_train]])
    test_split = non_wt.iloc[n_train:]

    name = f"rep-{replicate}"
    data = multidms.Data(
        train_split,
        alphabet=multidms.AAS_WITHSTOP_WITHGAP,
        reference=reference,
        assert_site_integrity=False,
        verbose=False,
        name=name,
    )
    data.condition_colors = cc
    train.append(data)
    test[name] = test_split

print(f"Train: {len(train)} datasets, Test: {len(test)} datasets")

## Fit CV models

In [ ]:
fit_params = {
    "fusionreg": fit["fusionreg_values"],
    "l2reg": [fit["l2reg"]],
    "beta0_ridge": [fit["beta0_ridge"]],
    "ge_type": [fit["ge_type"]],
    "tol": [fit.get("tol", 1e-6)],
    "maxiter": [fit["maxiter"]],
    "ge_kwargs": [fit["ge_kwargs"]],
    "cal_kwargs": [fit["cal_kwargs"]],
    "loss_kwargs": [fit["loss_kwargs"]],
    "warmstart": [fit["warmstart"]],
    "beta0_init": [fit.get("beta0_init", {})],
    "alpha_init": [fit.get("alpha_init", {})],
    "beta_clip_range": [tuple(fit.get("beta_clip_range", (-10, 10)))],
    "dataset": train,
}

_, _, models_cv = multidms.model_collection.fit_models(
    fit_params, gpu_ids=gpu_ids, n_processes=n_processes
)

for col in models_cv.columns:
    if models_cv[col].apply(lambda x: isinstance(x, dict)).any():
        models_cv[col] = models_cv[col].apply(str)

print(f"Fit {len(models_cv)} CV models")

In [ ]:
mc = multidms.model_collection.ModelCollection(models_cv)

# Filter test data for unseen mutations
filtered_test = {}
for name, test_df in test.items():
    model = mc.fit_models.query(f"dataset_name == '{name}'").iloc[0].model
    known_muts = set(model.data.mutations)

    def all_muts_known(subs):
        if pd.isna(subs) or subs == "":
            return True
        return all(m in known_muts for m in subs.split())

    mask = test_df["aa_substitutions"].apply(all_muts_known)
    filtered_test[name] = test_df[mask]
    n_dropped = (~mask).sum()
    if n_dropped > 0:
        print(f"{name}: dropped {n_dropped}/{len(test_df)} test variants with unseen mutations")

mc.add_eval_loss(filtered_test, overwrite=True)

In [ ]:
pickle.dump(models_cv, open(f"{output_dir}/cv_models.pkl", "wb"))
print(f"Saved {output_dir}/cv_models.pkl")

## Shrinkage trace plots

In [ ]:
# Get sparsity and correlation data from the full models
model_collection = multidms.ModelCollection(models)
chart, sparsity_df = model_collection.shift_sparsity(return_data=True, height_scalar=100)
chart, corr_df = model_collection.mut_param_dataset_correlation(width_scalar=200, return_data=True)
cross_validation_df = mc.loss_df()

saveas = "shrinkage_analysis_trace_plots_beta"
fig, ax = plt.subplots(3, figsize=[4.5, 7.5], sharex=True)

# Replicate correlation
iter_ax = ax[0]
sns.lineplot(
    data=(
        corr_df.query("mut_param.str.contains('shift')")
        .rename({"mut_param": "shift params"}, axis=1)
        .replace({"shift_Delta": "Delta", "shift_Omicron_BA2": "BA.2"})
        .assign(
            fusionreg=[f"{l:.1e}" for l in corr_df.query("mut_param.str.contains('shift')").fusionreg],
            correlation=lambda x: x.correlation ** 2,
        )
        .reset_index(drop=True)
    ),
    x="fusionreg", y="correlation", style="shift params",
    markers=True, ax=iter_ax, linewidth=3, color="black",
)
iter_ax.set_ylabel("rep1 v. rep2\nshift $(R^2)$")
iter_ax.legend(bbox_to_anchor=(1, 1), loc="upper left", frameon=False)

# Loss
iter_ax = ax[1]
sns.lineplot(
    data=cross_validation_df.query("condition=='total'").assign(
        lasso_strength=lambda x: x["fusionreg"].apply(lambda y: f"{y:.1e}")
    ),
    x="lasso_strength", y="loss", ax=iter_ax,
    hue="split", style="dataset_name",
    palette={"training": "slategrey", "validation": "#2CA02C"},
    markers=True, linewidth=3,
)
iter_ax.legend(bbox_to_anchor=(1, 1), loc="upper left", frameon=False)

# Sparsity
iter_ax = ax[2]
sns.lineplot(
    data=sparsity_df.rename({"dataset_name": "replicate", "mut_param": "shift params", "mut_type": "mutation type"}, axis=1)
    .replace({"shift_Delta": "Delta", "shift_Omicron_BA2": "BA.2"})
    .assign(
        fusionreg=[f"{l:.1e}" for l in sparsity_df.fusionreg],
        sparsity_percent=lambda x: x.sparsity * 100,
    ),
    x="fusionreg", y="sparsity_percent",
    hue="mutation type", style="replicate",
    palette={"nonsynonymous": "grey", "stop": "#E377C2"},
    markers=True, legend=True, ax=iter_ax, linewidth=3,
)
iter_ax.legend(bbox_to_anchor=(1, 1), loc="upper left", frameon=False)
iter_ax.set_xticklabels(iter_ax.get_xticklabels(), rotation=90, ha="center")
iter_ax.set_ylabel("sparsity\n$(\%\Delta=0)$")
iter_ax.set_xlabel("lasso regularization strength ($\lambda$)")

for axes in ax:
    axes.axvline(f"{chosen_lasso_strength:.1e}", color="grey", linewidth=10, alpha=0.35)

sns.despine(fig)
plt.tight_layout()
fig.savefig(f"{output_dir}/{saveas}.pdf", bbox_inches="tight")
plt.show()